# Jump to Any Timestep in One Line

**Topic:** Diffusion forward process &nbsp;·&nbsp; **Difficulty:** Medium

The diffusion forward process adds a little noise a thousand times. Training a
model that way, actually simulating 1000 steps to get one noisy sample, would
be hopeless. The saving grace is that composing Gaussians gives a Gaussian, so
`x_t` given `x_0` has a closed form and you can sample any timestep directly.

Verify the closed form against the step-by-step simulation, then look at what the
noise schedule does to the signal-to-noise ratio.

## The data (created in the next cell)

- `T = 1000` steps, linear beta schedule from `1e-4` to `0.02`.
- `x0` `(2000, 8)`: clean data.
- Timesteps of interest: `[1, 10, 100, 500, 999]`. Index `alpha_bar` with these
   numbers as given: `alpha_bar[1]`, `alpha_bar[10]`, `alpha_bar[100]`,
   `alpha_bar[500]`, `alpha_bar[999]`.

## Your task

With **NumPy only**:

1. `alphas = 1 - betas`, `alpha_bar = cumprod(alphas)`.
2. Closed form: `x_t = sqrt(alpha_bar[t]) * x0 + sqrt(1 - alpha_bar[t]) * eps`.
3. Simulate step by step and compare the *distribution*: for `t = 100`, run 100
   sequential steps on a copy of `x0` using `np.random.default_rng(123)`, and
   compare the empirical mean and variance against the closed form drawn with
   `np.random.default_rng(456)`. Those 100 steps consume `alphas[0]` through
   `alphas[99]`, whose product is `alpha_bar[99]`, so the closed form for that draw uses
   `alpha_bar[99]` and not `alpha_bar[100]`. Both seeds are needed: the grader compares your
   `sim_mean_gap` and `sim_var_gap` with the reference to a relative tolerance
   of 1e-5.
4. `snr`: `alpha_bar / (1 - alpha_bar)` at the timesteps of interest (an
   intermediate, graded through `snr_db`).
5. `t_half`: the smallest `t` where `alpha_bar[t] < 0.5`.
6. Set `answers` to a dict with `"alpha_bar_at"` (5 values), `"snr_db"` (5 values,
   `10 log10(snr)`), `"t_half"`, `"sim_mean_gap"`, `"sim_var_gap"`.

## Evaluation

Compared to the reference to a relative tolerance of 1e-5. The evaluation cell
checks the simulated and closed-form distributions agree and that SNR falls
monotonically.

In [ ]:
import numpy as np

rng = np.random.default_rng(4011)
T = 1000
betas = np.linspace(1e-4, 0.02, T)
x0 = rng.normal(0, 1, (2000, 8)) * 2.0 + 1.0
ts = [1, 10, 100, 500, 999]
print(f"T = {T}, beta from {betas[0]:g} to {betas[-1]:g}, x0 {x0.shape}")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("alpha_bar_at", [5], 2.9705652721401323, -1.2133414440043555, 1e-05),
    ("snr_db", [5], 17.789597659664892, -89.49477663716902, 1e-05),
    ("t_half", [], 259.0, -43.36644583695528, 1e-05),
    ("sim_mean_gap", [], 0.0019648394603593022, -0.0003289888186717534, 1e-05),
    ("sim_var_gap", [], 0.028049352777116177, -0.004696527945831892, 1e-05),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "snr_db" in answers and "sim_mean_gap" in answers and "sim_var_gap" in answers:
    _s = np.asarray(answers["snr_db"], dtype=float)
    _mono = bool(np.all(np.diff(_s) < 0))
    _agree = answers["sim_mean_gap"] < 0.05 and answers["sim_var_gap"] < 0.1
    print(f"  {'SNR falls monotonically':<24} {'ok' if _mono else 'wrong'}  "
          f"({np.round(_s, 1).tolist()} dB)")
    print(f"  {'closed form = simulation':<24} {'ok' if _agree else 'wrong'}  "
          f"(mean gap {answers['sim_mean_gap']:.4f}, var gap {answers['sim_var_gap']:.4f})")
    _ok = _ok and _mono and _agree

print("=" * 56)
if _ok:
    print("PASSED. Nice work!")
else:
    print("NOT PASSED yet. Revisit your solution and try again.")

## Solution

Each step is `x_t = sqrt(1-beta_t) x_{t-1} + sqrt(beta_t) eps`, and composing `t`
of them gives a Gaussian whose mean coefficient is the product of the
`sqrt(1-beta)` terms, that is `sqrt(alpha_bar_t)`, with the variances adding to
exactly `1 - alpha_bar_t`. Being able to jump straight to any `t` is what makes
training a random-timestep objective affordable. The SNR curve is the useful
diagnostic: it tells you which timesteps still carry signal, and it is why
schedule design (cosine, shifted, zero-terminal-SNR) matters as much as it
does.

```python
import numpy as np

alphas = 1.0 - betas
alpha_bar = np.cumprod(alphas)

idx = np.array(ts)
ab = alpha_bar[idx]
snr = ab / (1.0 - ab)
snr_db = 10.0 * np.log10(snr)

t_half = int(np.flatnonzero(alpha_bar < 0.5)[0])

r = np.random.default_rng(123)
x = x0.copy()
for t in range(100):
    eps = r.normal(0, 1, x.shape)
    x = np.sqrt(alphas[t]) * x + np.sqrt(betas[t]) * eps
sim_mean, sim_var = x.mean(), x.var()

r2 = np.random.default_rng(456)
eps = r2.normal(0, 1, x0.shape)
xc = np.sqrt(alpha_bar[99]) * x0 + np.sqrt(1 - alpha_bar[99]) * eps
cf_mean, cf_var = xc.mean(), xc.var()

answers = {
    "alpha_bar_at": ab,
    "snr_db": snr_db,
    "t_half": float(t_half),
    "sim_mean_gap": float(abs(sim_mean - cf_mean)),
    "sim_var_gap": float(abs(sim_var - cf_var)),
}
```